In [1]:
!pip install py-vncorenlp

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.9 MB/s eta 0:00:00
  Created wheel for py-vncorenlp: filename=py_vncorenlp-0.1.4-py3-none-any.whl size=4304 sha256=03e230e0bbbd4c7d01fcdb716b27d9208f879ad1df1b3915fcb6e21f5f44a4fd
  Stored in directory: /root/.cache/pip/wheels/db/e5/ff/f4a1b4ece36e8582db1ca71150a34e987e65df50c35974e9bb
Successfully built py-vncorenlp


In [2]:
import os
import shutil
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, MBartModel, AutoConfig
from tqdm.auto import tqdm
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

2026-01-11 20:14:46.504614: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768162486.687217      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768162486.740993      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768162487.147661      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768162487.147697      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768162487.147700      25 computation_placer.cc:177] computation placer alr

In [ ]:
class Config:
    TRAIN_PATH = '/kaggle/input/neudata/train_set.csv'
    VAL_PATH = '/kaggle/input/neudata/val_set.csv'
    
    MODEL_NAME = "vinai/bartpho-word"
    
    MAX_LEN = 1024         
    BATCH_SIZE = 4         
    ACCUMULATION_STEPS = 16  
    
    EPOCHS = 5
    LEARNING_RATE = 2e-5
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    RDR_PATH = '/kaggle/working/vncorenlp_fixed'

def setup_vncorenlp():
    should_download = True
    if os.path.exists(Config.RDR_PATH):
        if os.path.exists(os.path.join(Config.RDR_PATH, 'VnCoreNLP-1.2.jar')):
            should_download = False
        else:
            print("Phát hiện phiên bản cũ/lỗi. Đang xóa để tải bản chuẩn 1.2...")
            shutil.rmtree(Config.RDR_PATH)
    
    if should_download:
        print("Đang tải VnCoreNLP (v1.2)...")
        try:
            os.makedirs(Config.RDR_PATH, exist_ok=True)
            py_vncorenlp.download_model(save_dir=Config.RDR_PATH)
            print("✅ Tải thành công!")
        except Exception as e:
            print(f"❌ Lỗi tải VnCoreNLP: {e}")

setup_vncorenlp()

try:
    if 'rdrsegmenter' not in globals():
        rdrsegmenter = py_vncorenlp.VnCoreNLP(save_dir=Config.RDR_PATH, annotators=["wseg"])
        print("✅ RDRSegmenter đã sẵn sàng.")
    else:
        print("ℹ️ RDRSegmenter đang chạy (bỏ qua khởi tạo lại).")
except Exception as e:
    print(f"⚠️ Lỗi khởi tạo Java: {e}. Hãy Restart Kernel nếu cần.")

def segment_text(text):
    if not isinstance(text, str) or not text.strip(): return ""
    try:
        sentences = rdrsegmenter.word_segment(text)
        return " ".join(sentences)
    except:
        return text

import py_vncorenlp

Đang tải VnCoreNLP (v1.2)...
❌ Lỗi tải VnCoreNLP: name 'py_vncorenlp' is not defined
⚠️ Lỗi khởi tạo Java: name 'py_vncorenlp' is not defined. Hãy Restart Kernel nếu cần.


In [4]:
class MultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.texts = df['text'].values
        self.sentiments = df['sentiment'].values
        self.topics = df['classification'].values 

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        raw_text = str(self.texts[index])
        text = segment_text(raw_text)
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'sentiment_label': torch.tensor(self.sentiments[index], dtype=torch.long),
            'topic_label': torch.tensor(self.topics[index], dtype=torch.long)
        }

In [ ]:
class MultiTaskBARTpho(nn.Module):
    def __init__(self, n_sentiment, n_topic):
        super(MultiTaskBARTpho, self).__init__()
        print(f"Loading {Config.MODEL_NAME}...")
        self.bart = MBartModel.from_pretrained(Config.MODEL_NAME)

        self.bart.gradient_checkpointing_enable() 
        
        hidden_size = self.bart.config.hidden_size
        self.drop = nn.Dropout(p=0.1)
        
        self.sent_out = nn.Linear(hidden_size, n_sentiment)
        self.topic_out = nn.Linear(hidden_size, n_topic)

    def forward(self, input_ids, attention_mask):
        encoder_outputs = self.bart.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        last_hidden_state = encoder_outputs.last_hidden_state
        cls_output = last_hidden_state[:, 0, :] 
        output = self.drop(cls_output)
        return self.sent_out(output), self.topic_out(output)


In [ ]:
def validate_epoch(model, dataloader, device, w_sent=None, w_topic=None):
    model.eval() 
    total_loss = 0
    correct_sent = 0
    correct_topic = 0
    total_samples = 0
    
    crit_sent = nn.CrossEntropyLoss(weight=w_sent)
    crit_topic = nn.CrossEntropyLoss(weight=w_topic)
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating", leave=False):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            sent_lbl = batch['sentiment_label'].to(device)
            topic_lbl = batch['topic_label'].to(device)

            # Forward pass
            sent_logits, topic_logits = model(input_ids, mask)
            
            # 1. Calculate Loss
            loss_s = crit_sent(sent_logits, sent_lbl)
            loss_t = crit_topic(topic_logits, topic_lbl)
            loss = loss_s + loss_t
            total_loss += loss.item()
            
            # 2. Calculate Accuracy
            # Sentiment
            pred_sent = torch.argmax(sent_logits, dim=1)
            correct_sent += (pred_sent == sent_lbl).sum().item()
            
            # Topic
            pred_topic = torch.argmax(topic_logits, dim=1)
            correct_topic += (pred_topic == topic_lbl).sum().item()
            
            total_samples += sent_lbl.size(0)
            
    avg_loss = total_loss / len(dataloader)
    acc_sent = correct_sent / total_samples
    acc_topic = correct_topic / total_samples
    
    return avg_loss, acc_sent, acc_topic

In [ ]:
from torch.cuda.amp import autocast, GradScaler

def train_epoch(model, dataloader, optimizer, device, w_sent=None, w_topic=None):
    model.train()
    total_loss = 0
    
    crit_sent = nn.CrossEntropyLoss(weight=w_sent)
    crit_topic = nn.CrossEntropyLoss(weight=w_topic)
    
    scaler = GradScaler()
    
    optimizer.zero_grad() 
    
    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    
    for step, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        sent_lbl = batch['sentiment_label'].to(device)
        topic_lbl = batch['topic_label'].to(device)

        with autocast():
            sent_logits, topic_logits = model(input_ids, mask)
            loss_s = crit_sent(sent_logits, sent_lbl)
            loss_t = crit_topic(topic_logits, topic_lbl)
            
            loss = (loss_s + loss_t) / Config.ACCUMULATION_STEPS

        # Backward pass với scaler
        scaler.scale(loss).backward()
        
        # Chỉ cập nhật weights sau đủ số bước accumulation
        if (step + 1) % Config.ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # Optimizer step và update scaler
            scaler.step(optimizer)
            scaler.update()
            
            # Reset gradients
            optimizer.zero_grad()
        
        # Nhân lại loss với steps để hiển thị đúng giá trị thực
        current_loss = loss.item() * Config.ACCUMULATION_STEPS
        total_loss += current_loss
        
        del input_ids, mask, sent_lbl, topic_lbl, sent_logits, topic_logits, loss
        
        progress_bar.set_postfix({'loss': f"{current_loss:.4f}"})
        
    return total_loss / len(dataloader)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

def get_class_weights(y, device):
    class_weights = compute_class_weight(
        class_weight='balanced', 
        classes=np.unique(y), 
        y=y
    )
    return torch.tensor(class_weights, dtype=torch.float).to(device)

In [9]:
if __name__ == '__main__':
    # A. Load Data
    print("--- 1. Loading Data ---")
    try:
        df_train = pd.read_csv(Config.TRAIN_PATH)
        df_val = pd.read_csv(Config.VAL_PATH)
        print(f"Train size: {len(df_train)} | Val size: {len(df_val)}")
    except FileNotFoundError:
        print("❌ Không tìm thấy file csv. Sử dụng dummy data.")
        df_train = pd.DataFrame({'text': ['Demo'], 'sentiment': [0], 'classification': [0]})
        df_val = df_train.copy()

    # Config số lượng class
    n_sent = int(df_train['sentiment'].max()) + 1
    n_topic = int(df_train['classification'].max()) + 1
    print(f"Config: {n_sent} sentiment classes, {n_topic} topic classes.")

    # B. Tokenizer & Dataset
    print("\n--- 2. Preparing Dataset ---")
    tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
    
    train_ds = MultiTaskDataset(df_train, tokenizer, Config.MAX_LEN)
    val_ds = MultiTaskDataset(df_val, tokenizer, Config.MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)
    
    # C. Class weights
    w_sent = get_class_weights(df_train['sentiment'].values, Config.DEVICE)
    w_topic = get_class_weights(df_train['classification'].values, Config.DEVICE)

    # D. Model & Optimizer
    print("\n--- 3. Initializing BARTpho ---")
    model = MultiTaskBARTpho(n_sent, n_topic)
    model.to(Config.DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)
    
    # Biến để lưu model tốt nhất
    best_val_loss = float('inf')

    # E. Training Loop
    print("\n--- 4. Start Training ---")
    for epoch in range(Config.EPOCHS):
        print(f"\nEpoch {epoch+1}/{Config.EPOCHS}")
        
        # 1. Train
        train_loss = train_epoch(model, train_loader, optimizer, Config.DEVICE, w_sent, w_topic)
        
        # 2. Validate (Đã thêm phần này)
        val_loss, val_acc_sent, val_acc_topic = validate_epoch(model, val_loader, Config.DEVICE, w_sent, w_topic)
        
        print(f" -> Train Loss: {train_loss:.4f}")
        print(f" -> Val Loss  : {val_loss:.4f} | Sent Acc: {val_acc_sent:.4f} | Topic Acc: {val_acc_topic:.4f}")
        
        # 3. Save Best Model (Chỉ lưu nếu Val Loss giảm)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "bartpho_best.pth")
            print(" ✅ Saved Best Model")
        
        # Lưu checkpoint dự phòng mỗi epoch
        torch.save(model.state_dict(), f"bartpho_last_checkpoint.pth")

    print("\n✅ Training Completed!")

--- 1. Loading Data ---
Train size: 23048 | Val size: 3305
Config: 4 sentiment classes, 10 topic classes.

--- 2. Preparing Dataset ---


config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Validating:   0%|          | 0/827 [00:00<?, ?it/s]

 -> Train Loss: 0.9414
 -> Val Loss  : 1.6564 | Sent Acc: 0.7988 | Topic Acc: 0.7440

Epoch 5/5


Training:   0%|          | 0/5762 [00:00<?, ?it/s]

Validating:   0%|          | 0/827 [00:00<?, ?it/s]

 -> Train Loss: 0.7323
 -> Val Loss  : 1.7591 | Sent Acc: 0.7970 | Topic Acc: 0.7631

✅ Training Completed!


In [10]:
# --- MAPPING NHÃN (Lấy từ Notebook cũ của bạn) ---
ID2SENTIMENT = {
    0: "Neutral", 1: "Positive", 2: "Negative", 3: "Toxic"
}
# Đảo ngược để tra cứu nếu cần
SENTIMENT2ID = {v: k for k, v in ID2SENTIMENT.items()}

ID2TOPIC = {
    0: "Spam", 1: "News", 2: "Academic", 3: "Other", 4: "Service",
    5: "Jobs & Recruitment", 6: "Personal Affairs", 7: "Social Affairs",
    8: "Help & Share", 9: "Club & Events"
}
TOPIC2ID = {v: k for k, v in ID2TOPIC.items()}

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
import torch
from tqdm.auto import tqdm

print("--- Loading Test Data ---")
try:
    # Đường dẫn file test
    TEST_PATH = '/kaggle/input/neudata/test_set.csv'
    df_test = pd.read_csv(TEST_PATH)
    print(f"Test size: {len(df_test)}")
except Exception as e:
    print(f"❌ Lỗi load file test: {e}")
    df_test = pd.DataFrame({
        'text': ['Trường này tuyệt vời', 'Chán quá nghỉ học thôi'],
        'sentiment': [1, 2],
        'classification': [9, 6]
    })

test_ds = MultiTaskDataset(df_test, tokenizer, Config.MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False)


def evaluate_model(model, dataloader, device):
    model.eval() # Chuyển sang chế độ đánh giá
    
    true_sents, pred_sents = [], []
    true_topics, pred_topics = [], []
    
    print("Đang chạy đánh giá...")
    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            
            # Lấy nhãn gốc
            labels_s = batch['sentiment_label'].cpu().numpy()
            labels_t = batch['topic_label'].cpu().numpy()
            
            # Forward
            logits_s, logits_t = model(input_ids, mask)
            
            # Lấy nhãn dự đoán (Max probability)
            preds_s = torch.argmax(logits_s, dim=1).cpu().numpy()
            preds_t = torch.argmax(logits_t, dim=1).cpu().numpy()
            
            # Lưu kết quả
            true_sents.extend(labels_s)
            pred_sents.extend(preds_s)
            true_topics.extend(labels_t)
            pred_topics.extend(preds_t)
            
    return true_sents, pred_sents, true_topics, pred_topics


y_true_s, y_pred_s, y_true_t, y_pred_t = evaluate_model(model, test_loader, Config.DEVICE)

# --- BÁO CÁO CẢM XÚC (SENTIMENT) ---
print("\n" + "="*40)
print("KẾT QUẢ: CẢM XÚC (SENTIMENT)")
print("="*40)
print(f"Accuracy: {accuracy_score(y_true_s, y_pred_s):.4f}\n")
.
s_names = [ID2SENTIMENT[i] for i in sorted(ID2SENTIMENT.keys())]
print(classification_report(y_true_s, y_pred_s, target_names=s_names, zero_division=0))

# --- BÁO CÁO CHỦ ĐỀ (TOPIC) ---
print("\n" + "="*40)
print("KẾT QUẢ: CHỦ ĐỀ (TOPIC)")
print("="*40)
print(f"Accuracy: {accuracy_score(y_true_t, y_pred_t):.4f}\n")

t_names = [ID2TOPIC[i] for i in sorted(ID2TOPIC.keys())]
print(classification_report(y_true_t, y_pred_t, target_names=t_names, zero_division=0))

--- Loading Test Data ---
Test size: 6613
Đang chạy đánh giá...


  0%|          | 0/1654 [00:00<?, ?it/s]


KẾT QUẢ: CẢM XÚC (SENTIMENT)
Accuracy: 0.7916

              precision    recall  f1-score   support

     Neutral       0.91      0.81      0.85      4558
    Positive       0.61      0.72      0.66       834
    Negative       0.58      0.78      0.66      1046
       Toxic       0.86      0.84      0.85       175

    accuracy                           0.79      6613
   macro avg       0.74      0.79      0.76      6613
weighted avg       0.82      0.79      0.80      6613


KẾT QUẢ: CHỦ ĐỀ (TOPIC)
Accuracy: 0.7544

                    precision    recall  f1-score   support

              Spam       0.46      0.56      0.50        84
              News       0.41      0.64      0.50       183
          Academic       0.85      0.77      0.81      2104
             Other       0.86      0.82      0.84      2882
           Service       0.54      0.71      0.62       474
Jobs & Recruitment       0.58      0.67      0.62       163
  Personal Affairs       0.52      0.52      0.52    

In [12]:
def predict_sentence(text, model, tokenizer, device):
    model.eval()
    
    # 1. Tách từ (Rất quan trọng với BARTpho)
    text_segmented = segment_text(text)
    
    # 2. Tokenize
    encoding = tokenizer.encode_plus(
        text_segmented,
        max_length=Config.MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        logits_s, logits_t = model(input_ids, mask)
        
        # Lấy ID có điểm cao nhất
        id_s = torch.argmax(logits_s, dim=1).item()
        id_t = torch.argmax(logits_t, dim=1).item()
        
    return ID2SENTIMENT[id_s], ID2TOPIC[id_t]

# --- Test thử ---
sentence = "Bạn này chơi game Elden Ring trong lớp học này dẫn đến mình thiếu tập trung vì chỉ mãi xem bạn thôi!"
# Expect (neutral; personal affair)
print(f"Câu gốc: {sentence}")
sent, topic = predict_sentence(sentence, model, tokenizer, Config.DEVICE)
print(f"-> Cảm xúc: {sent}")
print(f"-> Chủ đề: {topic}")

Câu gốc: Bạn này chơi game Elden Ring trong lớp học này dẫn đến mình thiếu tập trung vì chỉ mãi xem bạn thôi!
-> Cảm xúc: Neutral
-> Chủ đề: Other


In [13]:
torch.save(model.state_dict(), f"bartpho_epoch_final.pth")
tokenizer.save_pretrained("./tokenizer_final")

('./tokenizer_final/tokenizer_config.json',
 './tokenizer_final/special_tokens_map.json',
 './tokenizer_final/vocab.txt',
 './tokenizer_final/bpe.codes',
 './tokenizer_final/added_tokens.json')